# **6장 주가 예측 시스템 구축**

## <span style="color: blue;">**1. 프롬프트**</span>

<span style="color: blue;">**다음 데이터를 이용해 주가 상승 확률을 실시간으로 추정하는 시스템을 만들고 싶다.**</span>

![Stock](https://raw.githubusercontent.com/pilsunchoi/images4/main/6-1.png)

자료: 미래에셋증권 모바일앱 스크린샷 2개 

## **Claude 아이디어**

이미지에서 호가창과 1분봉 차트 데이터를 확인했습니다. 주문 흐름 불균형, 가격 모멘텀, 거래량 등을 종합해 상승 확률을 실시간 추정하는 대시보드를 만들겠습니다. 업로드하신 호가창/차트 데이터를 기반으로 실시간 상승 확률 추정 시스템을 만들었습니다.

**추정 모델 구조** — 6개 팩터를 가중 합산합니다:

| 팩터 | 가중치 | 설명 |
|---|---|---|
| 호가 불균형 (전체) | 15% | 총 매수잔량 vs 매도잔량 비율 |
| 호가 불균형 (근접) | 25% | 현재가 인근 3호가 불균형 (가장 즉각적인 신호) |
| 가격 모멘텀 | 20% | 최근 5봉 가격 변화율 |
| 거래량 시그널 | 15% | 양봉 거래량 비중 + 거래량 확대 여부 |
| RSI 시그널 | 10% | 과매수/과매도 역행 신호 |
| 스프레드 압력 | 15% | 최우선 호가 간 스프레드 + 물량 비율 |

아래 대시보드에서 **"시뮬레이션 시작"** 버튼을 누르면 랜덤 틱을 생성하며 호가창·차트·확률이 1.2초마다 갱신됩니다. 실제 API를 연결하면 시뮬레이션 부분만 실시간 데이터 수신으로 교체하면 됩니다.

> 🔗 **주가 예측 시스템 이진 모델 대시보드** (https://claude.ai/public/artifacts/b52ce4f8-90bb-4782-9202-e778a5661dca)

![Stock](https://raw.githubusercontent.com/pilsunchoi/images4/main/6-2.png)

## <span style="color: blue;">**2. 가중치 설정 근거**</span>

## **Claude 답변**

좋은 질문입니다. 각 팩터의 가중치 설정 근거를 설명드리겠습니다.

**근접 호가 불균형 (25%) — 최고 가중치**

현재가 바로 위/아래 3호가의 매수·매도 잔량 차이는 초단기 가격 방향에 가장 직접적인 영향을 줍니다. 먼 호가의 물량은 취소·수정이 빈번하지만, 근접 호가는 실제 체결로 이어질 가능성이 높기 때문입니다. 업로드하신 데이터에서도 현재가(161,000) 바로 위 매도 잔량(161,100: 15,800)과 바로 아래 매수 잔량(160,900: 11,965)의 차이가 단기 방향성을 시사합니다.

**가격 모멘텀 (20%)**

1분봉 차트를 보면 18시 13분 158,800에서 19시 08분 161,600까지 급등 후 횡보하는 패턴입니다. 이런 모멘텀의 지속/소멸 여부가 다음 움직임을 결정하는 핵심 변수이므로 두 번째로 높게 잡았습니다.

**전체 호가 불균형 (15%) & 스프레드 압력 (15%)**

전체 호가 불균형(매도 254K vs 매수 159K)은 큰 그림의 수급을 보여주지만, 원거리 호가는 허수 주문이 많아 근접 호가보다 신뢰도가 낮습니다. 스프레드 압력도 보조적 확인 신호 역할이라 동일하게 15%로 설정했습니다.

**거래량 시그널 (15%)**

"가격은 거래량이 확인해 준다"는 원칙에 따라 양봉에 거래량이 실리는지를 봅니다. 다만 1분봉 단위에서는 노이즈가 크기 때문에 모멘텀보다는 낮게 잡았습니다.

**RSI 시그널 (10%) — 최저 가중치**

RSI는 후행 지표이고, 14봉 기준 계산이라 1분봉에서는 반응이 느립니다. 과매수/과매도 극단 영역에서만 의미 있는 역행 신호를 주므로 보조 역할로 10%를 부여했습니다.

---

솔직히 말씀드리면, 이 가중치는 일반적인 마켓 마이크로스트럭처 연구의 통념을 반영한 초기값입니다. 실제로 정밀하게 쓰려면 과거 데이터로 백테스트를 돌려 최적 가중치를 찾거나, 로지스틱 회귀/XGBoost 같은 모델로 학습시키는 것이 훨씬 정확합니다. 원하시면 가중치를 UI에서 슬라이더로 직접 조절할 수 있게 수정해 드릴 수도 있습니다.

## <span style="color: blue;">**3. (1-상승확률)은 하락확률?**</span>

## **Claude 답변**

엄밀히 말하면 아닙니다.

가격의 다음 움직임은 상승, 보합, 하락 세 가지이기 때문에 `1 - P(상승)` = `P(보합) + P(하락)`입니다. 현재 모델은 이걸 구분하지 않고 이진 스코어로 처리하고 있어서, 예를 들어 상승확률 40%가 나왔을 때 "하락 60%"인지 "보합 가능성이 큰 건지" 구분이 안 되는 한계가 있습니다.

개선하려면 세 가지 방향이 있습니다.

첫째, 3분류로 바꾸는 방법입니다. 상승/보합/하락 각각의 확률을 추정해서 합이 100%가 되도록 softmax 형태로 출력하는 겁니다. 예: 상승 35%, 보합 40%, 하락 25%.

둘째, 임계값을 두는 방법입니다. 현재가 대비 ±0.1% 이내 변동을 보합으로 정의하고, 그 범위를 벗어나는 방향의 확률만 상승/하락으로 분류합니다.

셋째, 기대수익률로 전환하는 방법입니다. 확률 대신 "향후 N분간 예상 변동폭"을 추정하면 방향과 크기를 동시에 표현할 수 있습니다.

3분류 모델로 업그레이드할까요?

## <span style="color: blue;">**4. 모델 업그레이드: 3분류**</span>

## **Claude 작업**

전체 파일을 3분류 모델로 재작성하겠습니다. 핵심 변경사항은 softmax 기반 3분류 확률, 보합 영역 임계값 도입, 3색 확률 추이 차트입니다.3분류 모델로 업그레이드했습니다. 주요 변경사항은 다음과 같습니다.

**모델 구조 변경**

기존 이진 스코어(0~100%) 대신, 6개 팩터의 가중합으로 방향성 점수(directional score)를 먼저 산출하고, 여기에 변동성(volatility) 지표를 결합해 3개 로짓을 만듭니다. 상승 로짓은 방향성이 양수일 때, 하락 로짓은 음수일 때 커지고, 보합 로짓은 변동성이 낮고 방향성이 약할수록 강화됩니다. 이 3개 로짓에 Softmax를 적용해 합계 100%인 확률 분포로 변환합니다.

**보합 판정 로직**

보합 강도 = `(1 - 변동성) × 0.7 + (1 - 방향성 강도) × 0.3` 공식으로, 최근 캔들의 변동폭이 작으면서 팩터들이 상충해 방향성이 약한 구간에서 보합 확률이 자연스럽게 올라갑니다.

**UI 변경**

게이지 차트를 3분류 바 차트로 교체하고, 확률 추이 차트도 3개 라인(초록/노랑/빨강)으로 분리했습니다. 방향성 점수, 변동성 레벨, RSI를 별도 메트릭 카드로 표시합니다.


> 🔗 **주가 예측 시스템 3분류 모델 대시보드** (https://claude.ai/public/artifacts/159d4993-6fdd-4468-8802-d5d25f6026f8)

![Stock](https://raw.githubusercontent.com/pilsunchoi/images4/main/6-3.png)